# 03B — Pemodelan BiLSTM pada Data Baru (v2): 3 Skenario Simulasi Ketimpangan
Notebook ini menguji ketahanan model BiLSTM terhadap 3 rasio ketimpangan data latih buatan:
1. **Skenario 1:1:1** (Seimbang Sempurna: 33.3% Negatif, 33.3% Netral, 33.3% Positif)
2. **Skenario 6:3:1** (Ketimpangan Moderat: 60% Negatif, 30% Positif, 10% Netral)
3. **Skenario 8:1:1** (Ketimpangan Ekstrem / Long-tail: 80% Negatif, 10% Positif, 10% Netral)

Setiap skenario dievaluasi terhadap data uji empiris terkunci (n = 1.730) dengan 5 varian balancing (Baseline, CW, ROS, RUS, SMOTE).


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dropout, Dense
from sklearn.metrics import classification_report, accuracy_score, f1_score

if Path.cwd().name == 'notebooks':
    os.chdir('..')

# Cek file skenario simulasi
for sc in ['111', '631', '811']:
    p = f'Data/simulated/scenario_{sc}.csv'
    if Path(p).exists():
        df_sc = pd.read_csv(p)
        counts = df_sc['label'].value_counts().to_dict()
        print(f'Scenario {sc}: {len(df_sc)} rows | Distribusi: {counts}')


Scenario 111: 3000 rows | Distribusi: {1: 1000, 0: 1000, 2: 1000}
Scenario 631: 5000 rows | Distribusi: {0: 3000, 2: 1500, 1: 500}


Scenario 811: 4000 rows | Distribusi: {0: 3200, 1: 400, 2: 400}


## Hasil Rangkuman Simulasi Ketimpangan BiLSTM
| Skenario | Model Baseline F1 | ROS Winner F1 | Delta Pemulihan F1 | Diagnosa Ilmiah |
| :--- | :---: | :---: | :---: | :--- |
| **1:1:1** | 58,16% | 56,19% | -1,97 pp | Memaksa data seimbang membuat model kehilangan prior alami populasi |
| **6:3:1** | 57,16% | **59,95%** | **+2,79 pp** | ROS meningkatkan stabilitas deteksi minoritas |
| **8:1:1** | 45,97% | **58,51%** | **+12,54 pp** | Terjadi **Majority Collapse** pada baseline; teknik penyeimbangan **WAJIB** diterapkan |
